# Evaluation 05 — Probability calibration and reliability (real/synthetic separated)

Requires **six probability columns** (`p_acne`, `p_fungal_infection`, `p_vitiligo`, `p_hyperpigmentation`, `p_skin_cancer`, `p_normal`) from an inference notebook. Never calibrate/choose hyperparameters on the final test set.

**Saved evidence:** `notebook_console.log` captures execution/tracebacks, `run_log.jsonl` records start and completed artifact SHA-256 checksums, and each analysis writes its reported CSV/JSON or graphic files into the configured Kaggle output directory. Keep per-image records and raw console text private; export only reviewed aggregate reports to GitHub.


In [ ]:
# Common Kaggle bootstrap: either attach DermSynReal source as Kaggle Input,
# or set CLONE_REPO_IF_MISSING=True and enable Kaggle Internet.
from pathlib import Path
import subprocess,sys,os,json
REPO_ROOT=Path("/kaggle/input/EDIT_REPO_DATASET/DermSynReal")
CLONE_REPO_IF_MISSING=True
EXPECTED_GIT_SHA=""  # Recommended: set a full pinned SHA for reproducible final runs.
if not (REPO_ROOT/"src/dermsynreal/evaluation.py").is_file():
    if not CLONE_REPO_IF_MISSING:
        raise FileNotFoundError("Attach the repository and change REPO_ROOT, or enable Git clone.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if EXPECTED_GIT_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("For pinned SHA verification attach a Git checkout or enable clone")
    actual=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual!=EXPECTED_GIT_SHA:
        raise ValueError(f"Repository SHA {actual} differs from pinned SHA {EXPECTED_GIT_SHA}")
elif (REPO_ROOT/".git").is_dir():
    print("Current Git SHA:",subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip())
    print("WARNING: pin EXPECTED_GIT_SHA for final research outputs")
sys.path.insert(0,str(REPO_ROOT/"src"))
print("DermSynReal source:",REPO_ROOT)


In [ ]:
PREDICTIONS_CSV="/kaggle/input/EDIT_PREDICTION_ARTIFACT/final_test_predictions.csv"
OUTPUT_DIR="/kaggle/working/dermsynreal_eval_05"
ECE_BINS=10
RUN_CALIBRATION=False

In [ ]:
# Preserve a complete Kaggle notebook console and structured start event.
# This log can contain image file paths or case notes; keep it out of public GitHub.
from dermsynreal.kaggle_runs import start_notebook_logging,append_run_log
EVALUATION_OUTPUT=Path(OUTPUT_DIR)
PRIVATE_EVALUATION_LOG=start_notebook_logging(EVALUATION_OUTPUT,
    kind='evaluation_05',config={
        'notebook':'05_Probabilities_Calibration_and_Errors.ipynb','data_verified':False,
        'run_enabled':bool(RUN_CALIBRATION)},repo_root=REPO_ROOT)
print('Saved notebook console:',PRIVATE_EVALUATION_LOG)


In [ ]:
from dermsynreal.calibration import calibration_by_domain
if RUN_CALIBRATION:
    summary=calibration_by_domain(PREDICTIONS_CSV,bins=ECE_BINS)
    output=Path(OUTPUT_DIR);output.mkdir(parents=True,exist_ok=True)
    (output/"calibration_by_domain.json").write_text(json.dumps(summary,indent=2)+"\n")
    import matplotlib.pyplot as plt
    for source,item in summary.items():
        print(source,{k:item[k] for k in ("n","ece","multiclass_brier","nll")})
        b=[x for x in item["bins"] if x["n"]]
        plt.figure(figsize=(5,5))
        plt.plot([0,1],[0,1],"--",label="ideal")
        plt.plot([x["mean_confidence"] for x in b],[x["accuracy"] for x in b],
                 "o-",label=f"{source} ({item['n']} images)")
        plt.xlim(0,1);plt.ylim(0,1);plt.xlabel("Confidence");plt.ylabel("Accuracy")
        plt.title(f"Reliability — {source} test")
        plt.legend();plt.tight_layout()
        plt.savefig(output/f"reliability_{source}.png",dpi=170);plt.show()
else:
    print("Attach actual probability-based model predictions and set RUN_CALIBRATION=True.")


In [ ]:
# Error analysis by domain and true class (no facial image identities displayed).
if RUN_CALIBRATION:
    from dermsynreal.evaluation import read_predictions
    import pandas as pd
    entries=read_predictions(PREDICTIONS_CSV)
    error_rows=[]
    for row in entries:
        error_rows.append({"source":row["source"],"true_label":row["true_label"],
                           "pred_label":row["pred_label"],
                           "incorrect":int(row["true_label"]!=row["pred_label"]),
                           "image_id":row["image_id"],
                           "dataset_source":row.get("dataset_source","")})
    errors_df=pd.DataFrame(error_rows)
    by_class=errors_df.groupby(["source","true_label"]).agg(
        support=("incorrect","size"),incorrect=("incorrect","sum")
    ).reset_index()
    by_class["error_rate"]=by_class["incorrect"]/by_class["support"]
    by_class.to_csv(Path(OUTPUT_DIR)/"domain_class_error_rates.csv",index=False)
    errors_df.loc[errors_df["incorrect"]==1].to_csv(
        Path(OUTPUT_DIR)/"misclassified_image_ids.csv",index=False)
    display(by_class)
    print("Error files written; review systematically by class and real data source.")


ECE is a descriptive bin-dependent diagnostic, not a substitute for clinical validation. Test results must not be used to tune calibration temperature or binning.

In [ ]:
# Record checksums only for files actually created by this executed run.
if RUN_CALIBRATION:
    _files=[Path(OUTPUT_DIR)/'calibration_by_domain.json',Path(OUTPUT_DIR)/'domain_class_error_rates.csv']
    _log=append_run_log(EVALUATION_OUTPUT,kind='evaluation_05_finished',
        config={'notebook':'05_Probabilities_Calibration_and_Errors.ipynb','status':'completed_artifacts_written'},
        artifacts=[p for p in _files if p.is_file()],repo_root=REPO_ROOT)
    print('Evidence hashes:',_log['artifacts_sha256'])
